# ML2025 Homework 1 - Retrieval Augmented Generation with Agents

## Environment Setup

In this section, we install the necessary python packages and download model weights of the quantized version of LLaMA 3.1 8B. Also, download the dataset. Note that the model weight is around 8GB.

In [1]:
!python3 -m pip install --no-cache-dir llama-cpp-python==0.3.4 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!python3 -m pip install googlesearch-python bs4 charset-normalizer requests-html lxml_html_clean

from pathlib import Path
if not Path('./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf').exists():
    !wget https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
if not Path('./public.txt').exists():
    !wget https://www.csie.ntu.edu.tw/~ulin/public.txt
if not Path('./private.txt').exists():
    !wget https://www.csie.ntu.edu.tw/~ulin/private.txt

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.2/445.2 MB 192.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 14.1
    Uninstalling websockets-14.1:
      Successfully uninstalled websockets-14.1
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.3.0
    Uninstalling urllib3-2.3.0:
      Successfully uninstalled urllib3-2.3.0
ERROR: pip's depen

In [2]:
import torch
if not torch.cuda.is_available():
    raise Exception('You are not using the GPU runtime. Change it first or you will suffer from the super slow inference speed!')
else:
    print('You are good to go!')

You are good to go!


## Prepare the LLM and LLM utility function

By default, we will use the quantized version of LLaMA 3.1 8B. you can get full marks on this homework by using the provided LLM and LLM utility function. You can also try out different LLM models.

In the following code block, we will load the downloaded LLM model weights onto the GPU first.
Then, we implemented the generate_response() function so that you can get the generated response from the LLM model more easily.

You can ignore "llama_new_context_with_model: n_ctx_per_seq (16384) < n_ctx_train (131072) -- the full capacity of the model will not be utilized" warning.

In [3]:
from llama_cpp import Llama

# Load the model onto GPU
llama3 = Llama(
    "./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf",
    verbose=False,
    n_gpu_layers=-1,
    n_ctx=16384,    # This argument is how many tokens the model can take. The longer the better, but it will consume more memory. 16384 is a proper value for a GPU with 16GB VRAM.
)

def generate_response(_model: Llama, _messages: str) -> str:
    '''
    This function will inference the model with given messages.
    '''
    _output = _model.create_chat_completion(
        _messages,
        stop=["<|eot_id|>", "<|end_of_text|>"],
        max_tokens=512,    # This argument is how many tokens the model can generate.
        temperature=0,      # This argument is the randomness of the model. 0 means no randomness. You will get the same result with the same input every time. You can try to set it to different values.
        repeat_penalty=1.1, #原来的惩罚值2.0太高了 模型为了不重复字开始胡言乱语
    )["choices"][0]["message"]["content"]
    return _output

llama_new_context_with_model: n_ctx_per_seq (16384) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


## Search Tool

The TA has implemented a search tool for you to search certain keywords using Google Search. You can use this tool to search for the relevant **web pages** for the given question. The search tool can be integrated in the following sections.

In [4]:
from typing import List
from googlesearch import search as _search
from bs4 import BeautifulSoup
from charset_normalizer import detect
import asyncio
from requests_html import AsyncHTMLSession
import urllib3
urllib3.disable_warnings()

async def worker(s:AsyncHTMLSession, url:str):
    try:
        header_response = await asyncio.wait_for(s.head(url, verify=False), timeout=10)
        if 'text/html' not in header_response.headers.get('Content-Type', ''):
            return None
        r = await asyncio.wait_for(s.get(url, verify=False), timeout=10)
        return r.text
    except:
        return None

async def get_htmls(urls):
    session = AsyncHTMLSession()
    tasks = (worker(session, url) for url in urls)
    return await asyncio.gather(*tasks)

async def search(keyword: str, n_results: int=3) -> List[str]:
    '''
    This function will search the keyword and return the text content in the first n_results web pages.
    Warning: You may suffer from HTTP 429 errors if you search too many times in a period of time. This is unavoidable and you should take your own risk if you want to try search more results at once.
    The rate limit is not explicitly announced by Google, hence there's not much we can do except for changing the IP or wait until Google unban you (we don't know how long the penalty will last either).
    '''
    keyword = keyword[:100]
    # First, search the keyword and get the results. Also, get 2 times more results in case some of them are invalid.
    results = list(_search(keyword, n_results * 2, lang="zh", unique=True))
    # Then, get the HTML from the results. Also, the helper function will filter out the non-HTML urls.
    results = await get_htmls(results)
    # Filter out the None values.
    results = [x for x in results if x is not None]
    # Parse the HTML.
    results = [BeautifulSoup(x, 'html.parser') for x in results]
    # Get the text from the HTML and remove the spaces. Also, filter out the non-utf-8 encoding.
    results = [''.join(x.get_text().split()) for x in results if detect(x.encode()).get('encoding') == 'utf-8']
    # Return the first n results.
    return results[:n_results]

In [5]:
!pip uninstall duckduckgo_search -y
!pip install -U ddgs duckduckgo-search

import asyncio
import random

# 兼容新舊套件名稱的聰明載入法
try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS

# 結合 DuckDuckGo 的穩定性與 TA 的爬蟲深度
async def search(keywords: str, n_results: int = 2) -> list[str]:
    print(f"🔍 搜尋網址: {keywords}")
    await asyncio.sleep(random.uniform(2.0, 4.0)) 
    try:
        def _sync_search():
            from ddgs import DDGS
            with DDGS() as ddgs:
                return list(ddgs.text(keywords, max_results=n_results))
        results = await asyncio.to_thread(_sync_search)
        if not results: return []
        
        # 提取網址並呼叫 TA 的爬蟲
        urls = [res.get('href') for res in results if res.get('href')]
        raw_htmls = await get_htmls(urls)
        
        final_texts = []
        for html in raw_htmls:
            if html:
                soup = BeautifulSoup(html, 'html.parser')
                text = ''.join(soup.get_text().split())
                final_texts.append(text)
        return final_texts
    except:
        return []

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 67.9 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7


## Test the LLM inference pipeline

In [6]:
# You can try out different questions here.
test_question='請問誰是 周杰倫？'

messages = [
    {"role": "system", "content": "你是 LLaMA-3.1-8B，是用來回答問題的 AI。使用中文時只會使用繁體中文來回問題。"},    # System prompt
    {"role": "user", "content": test_question}, # User prompt
]

print(generate_response(llama3, messages))

周杰倫是一位臺灣的歌手、詞曲作家、音樂製作人和導演。他出生於台北，1985年加入百代唱片（現為環球音樂）成為旗下藝人。周杰倫以其創新的音樂風格和多樣化的作品獲得廣泛認可，他的歌曲經常融合了不同的音樂類型，如流行、搖滾、電子等。

周杰倫的音樂風格被稱為「新民謠」，他在1999年推出首張專輯《Jay》後，迅速成長為臺灣樂壇的一顆巨星。他以其優秀的歌曲創作和演唱能力獲得了無數獎項和榮譽，包括多次獲得金曲獎、風雲榜等獎項。

除了音樂以外，周杰倫還是一位有才華的導演，他曾經執導過多部音樂錄影帶和電影。他的作品常常探討人生、愛情、家庭等題材，深受觀眾喜愛。他也是臺灣最具影響力的藝人之一，被譽為「世紀偶像」。


## Agents

The TA has implemented the Agent class for you. You can use this class to create agents that can interact with the LLM model. The Agent class has the following attributes and methods:
- Attributes:
    - role_description: The role of the agent. For example, if you want this agent to be a history expert, you can set the role_description to "You are a history expert. You will only answer questions based on what really happened in the past. Do not generate any answer if you don't have reliable sources.".
    - task_description: The task of the agent. For example, if you want this agent to answer questions only in yes/no, you can set the task_description to "Please answer the following question in yes/no. Explanations are not needed."
    - llm: Just an indicator of the LLM model used by the agent.
- Method:
    - inference: This method takes a message as input and returns the generated response from the LLM model. The message will first be formatted into proper input for the LLM model. (This is where you can set some global instructions like "Please speak in a polite manner" or "Please provide a detailed explanation".) The generated response will be returned as the output.

In [7]:
class LLMAgent():
    def __init__(self, role_description: str, task_description: str, llm:str="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"):
        self.role_description = role_description   # Role means who this agent should act like. e.g. the history expert, the manager......
        self.task_description = task_description    # Task description instructs what task should this agent solve.
        self.llm = llm  # LLM indicates which LLM backend this agent is using.
    def inference(self, message:str) -> str:
        if self.llm == 'bartowski/Meta-Llama-3.1-8B-Instruct-GGUF': # If using the default one.
            # TODO: Design the system prompt and user prompt here.
            # Format the messsages first.
            messages = [
                {"role": "system", "content": f"{self.role_description}\nYou can use English or other languages as needed. However, whenever you output Chinese, you must strictly use Traditional Chinese (繁體中文) ONLY. Do not use Simplified Chinese."},  # Hint: you may want the agents to speak Traditional Chinese only.
                {"role": "user", "content": f"{self.task_description}\n\n<user_input>\n{message}\n</user_input>"}, # Hint: you may want the agents to clearly distinguish the task descriptions and the user messages. A proper seperation text rather than a simple line break is recommended.
            ]
            return generate_response(llama3, messages)
        else:
            # TODO: If you want to use LLMs other than the given one, please implement the inference part on your own.
            return ""

TODO 1: Design the role description and task description for each agent.

In [8]:
# TODO: Design the role and task description for each agent.

# This agent may help you filter out the irrelevant parts in question descriptions.
question_extraction_agent = LLMAgent(
    role_description="你是提問優化專家，擅長將使用者的輸入重新整理成一個結構清晰、細節完整的問句。",
    task_description="""請將使用者的輸入重新整理成一個精煉的問句。
【絕對規則】
1. 絕對不能遺漏原句中的任何專有名詞、人名、地名、數字或條件限制。
2. 只輸出問句本身，絕對不要包含任何開場白或多餘的文字。"""
)

# This agent may help you extract the keywords in a question so that the search tool can find more accurate results.
keyword_extraction_agent = LLMAgent(
    role_description="你是頂尖的搜尋引擎優化（SEO）與實體識別專家。",
    task_description="""請從使用者的問題中，提取出最關鍵的「專有名詞」、「人名」或「核心概念」作為搜尋關鍵字。
【絕對規則】
1. 最多不超過 3 個關鍵字。
2. 只輸出關鍵字，用「空格」隔開，絕對不要使用逗號或任何標點符號。
3. 如果問題明顯與台灣相關（如台大、某國小、NCC），請自行在關鍵字中加上「台灣」。"""
)

# This agent is the core component that answers the question.
qa_agent = LLMAgent(
    role_description="你是 LLaMA-3.1-8B，一位精準、嚴謹的知識問答助手。You must strictly use Traditional Chinese (繁體中文) ONLY.",
    task_description="""請根據【參考資訊】回答【使用者問題】。如果參考資訊沒有提及，請結合你的內建知識。
【強制回答格式】
你必須嚴格遵守以下兩段式格式輸出：
分析：(簡短寫下你的推理過程)
答案：(寫下最終答案，請直接回答重點，不要重複使用者的問題，也不需要完整的解釋句子)"""
)

## RAG pipeline

TODO 2: Implement the RAG pipeline.

Please refer to the homework description slides for hints.

Also, there might be more heuristics (e.g. classifying the questions based on their lengths, determining if the question need a search or not, reconfirm the answer before returning it to the user......) that are not shown in the flow charts. You can use your creativity to come up with a better solution!

- Naive approach (simple baseline)

    ![](https://www.csie.ntu.edu.tw/~ulin/naive.png)

- Naive RAG approach (medium baseline)

    ![](https://www.csie.ntu.edu.tw/~ulin/naive_rag.png)

- RAG with agents (strong baseline)

    ![](https://www.csie.ntu.edu.tw/~ulin/rag_agent.png)

In [9]:
import asyncio

async def pipeline(question: str) -> str:
    # TODO: Implement your pipeline.
    # Currently, it only feeds the question directly to the LLM.
    # You may want to get the final results through multiple inferences.
    # Just a quick reminder, make sure your input length is within the limit of the model context window (16384 tokens), you may want to truncate some excessive texts.
    
    # 1. 問題優化 Agent
    refined_question = await asyncio.to_thread(question_extraction_agent.inference, question)
    
    # 2. 關鍵字 Agent (直接用空格分隔了，不用再取代逗號)
    keywords_str = await asyncio.to_thread(keyword_extraction_agent.inference, refined_question)
    clean_keywords = keywords_str.strip()
    
    # 3. 搜尋 (抓 5 篇 DuckDuckGo 摘要，搭配 1.5~3 秒防封鎖)
    raw_results_list = await search(clean_keywords, n_results=5)
    
    # 组合搜索结果字符串
    if not raw_results_list:
        search_results_text = "沒有找到相關網頁資訊。"
    else:
        search_results_text = "\n\n".join(raw_results_list)

    # 新增：字数截断保护机制 (防止 70,000+ Token 撑爆内存)
    max_length = 10000 
    if len(search_results_text) > max_length:
        search_results_text = search_results_text[:max_length] + "\n...(資訊過長，為保護模型記憶體已自動截斷)"

    # 4. 組合 Prompt
    combined_input = (
        f"【參考資訊】\n{search_results_text}\n\n"
        f"【使用者問題】\n{refined_question}"
    )

    # 5. QA Agent 進行思維鏈推理
    final_response = await asyncio.to_thread(qa_agent.inference, combined_input)
    
    # 6. 程式自動萃取純答案，保證輸出乾淨符合助教評分腳本
    if "答案：" in final_response:
        # 切割字串，只拿「答案：」後面的內容
        final_answer = final_response.split("答案：")[-1].strip()
    else:
        # 如果模型沒有照格式，就拿最後一行的結果
        final_answer = final_response.strip().split('\n')[-1]
        
    return final_answer

## Answer the questions using your pipeline!

Since Colab has usage limit, you might encounter the disconnections. The following code will save your answer for each question. If you have mounted your Google Drive as instructed, you can just rerun the whole notebook to continue your process.

In [10]:
from pathlib import Path

# Fill in your student ID first.
STUDENT_ID = "2024311169"

STUDENT_ID = STUDENT_ID.lower()
with open('./public.txt', 'r') as input_f:
    questions = input_f.readlines()
    questions = [l.strip().split(',')[0] for l in questions]
    for id, question in enumerate(questions, 1):
        if Path(f"./{STUDENT_ID}_{id}.txt").exists():
            continue
        answer = await pipeline(question)
        answer = answer.replace('\n',' ')
        print(id, answer)
        with open(f'./{STUDENT_ID}_{id}.txt', 'w') as output_f:
            print(answer, file=output_f)

with open('./private.txt', 'r') as input_f:
    questions = input_f.readlines()
    for id, question in enumerate(questions, 31):
        if Path(f"./{STUDENT_ID}_{id}.txt").exists():
            continue
        answer = await pipeline(question)
        answer = answer.replace('\n',' ')
        print(id, answer)
        with open(f'./{STUDENT_ID}_{id}.txt', 'a') as output_f:
            print(answer, file=output_f)

🔍 搜尋網址: 虎山雄風飛揚 台灣校歌
1 雖然我找不到直接證據表明「虎山雄風飛揚」是哪間學校的校歌，但根據台灣民歌運動的發展歷史和相關文獻，似乎有幾個可能的候選學校。其中一個可能性是淡江大學，因為淡江大學曾經是民歌運動的一個重要據點，並且有一些著名的民歌創作者與該校有關。然而，我們需要更多資訊來確認這一點。
🔍 搜尋網址: NCC 台灣境外郵購審查費
2 無法提供具體數額，因為相關資訊未被提及。
🔍 搜尋網址: Steve Jobs 台灣
3 史蒂夫·喬布斯
🔍 搜尋網址: 台灣大學進階英文免修申請規定 TOEFL iBT 分數
4 無法提供具體的分數要求，因為資訊不足。
🔍 搜尋網址: 觸地 try 可得 7 分， Rugby Union
5 5分
🔍 搜尋網址: 卑南族 台灣
6 台東縣
🔍 搜尋網址: 熊仔 台大 碩士指導教授
7 熊仔
🔍 搜尋網址: 麥克斯韋
8 詹姆斯·克拉克·馬克斯韋爾（James Clerk Maxwell）
🔍 搜尋網址: 國立臺灣史前文化博物館 台鐵車站 台灣
9 臺東車站
🔍 搜尋網址: 20與30的和是多少 

關鍵字：數學 算術
10 (寫下最終答案，請直接回答重點，不要重複使用者的問題，也不需要完整的解釋句子) 50
🔍 搜尋網址: 達拉斯獨行俠隊 Luka Doncic NBA
11 洛杉磯湖人隊
🔍 搜尋網址: 美國總統大選 台灣
12 尚未確定，川普暫時領先。
🔍 搜尋網址: Meta Llama-3.2 台灣
13 1B
🔍 搜尋網址: 學生停修課程規定 台灣教育部
14 3學分
🔍 搜尋網址: DeepSeek
15 High-Flyer
🔍 搜尋網址: NBA 台灣
16 無法確定。
🔍 搜尋網址: 碳原子 三鍵化合物
17 炔烴。
🔍 搜尋網址: 圖靈計算機科學
18 艾倫·圖靈
🔍 搜尋網址: 臺灣玄天上帝 台灣進香中心
19 南投縣
🔍 搜尋網址: 微軟 Windows 台灣
20 微軟公司。
🔍 搜尋網址: 官將首的起源 台灣廟宇
21 新莊地藏庵
🔍 搜尋網址: 咒邪神
22 邪神厲勿邪。
🔍 搜尋網址: 短暫交會旅程 台灣歌唱團體
23 《因果》
🔍 搜尋網址: 卑南族 台灣部落
24 任何一個卑南八社的部落都有可能舉辦聯合年祭。
🔍 搜尋網址: 輝達 GeForce RTX系列 台灣
25 RT

In [11]:
# Combine the results into one file.
with open(f'./{STUDENT_ID}.txt', 'w') as output_f:
    for id in range(1,91):
        with open(f'./{STUDENT_ID}_{id}.txt', 'r') as input_f:
            answer = input_f.readline().strip()
            print(answer, file=output_f)